# Survey speed vs. image quality: static PNG

Matplotlib version of the figure built in `04_interactive.ipynb`, from the same cached tables.
6 x 4 in, 12 pt axis labels, `TRANSPARENT` background switch (set to white while iterating; flip
back to transparent for delivery), Rubin colour palette (`ohlp/style.py`).

`PERIOD = "early_ops"` (default) draws a **single summary point** for Early Ops, 24 Oct–30 Nov 2025,
i.e. the nights before the Dec 2025–Mar 2026 sequence of downtimes (water condensation, Dynalene loss,
vacuum leak, earthquake; see `nightly_survey_speed/plots/downtime_summary.md`), with its nights as context.
`PERIOD = "early_ops_vs_lsst"` adds a **second point** for the LSST on-sky stretch, 29 Jun–13 Jul 2026
(season start to the last night with survey visits before the storm), in purple against the Early Ops teal,
with both nightly clouds. Each window is described in the legend
(`LABELS_IN_LEGEND`); set it to `False` for labels next to the points instead.
The other values reuse the tables written by `01_data_and_binning.ipynb`.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import warnings

import pandas as pd

warnings.simplefilter("ignore", FutureWarning)
sys.path.insert(0, ".")
from ohlp import binning, data, static, reference as ref
from ohlp.style import rp

In [ ]:
# Parameters (must match 01_data_and_binning.ipynb)
DAY_OBS_MIN = ref.DAY_OBS_EARLY_OPS_START
DAY_OBS_MAX = 20261014
PERIOD = "early_ops_vs_lsst"  # "early_ops" (single Oct+Nov 2025 point), "early_ops_vs_lsst" (adds the LSST point), "window", "quarter" or "month"
IQ_COL = "idiq_aos_cam"
FIGSIZE = (6, 4)  # inches
LABEL_SIZE = 12  # pt, axis labels
TRANSPARENT = True  # False = white background (easier on a dark screen); True for delivery
TITLE = None  # None for no title; set per PERIOD below

# Single Early Ops window: first survey night to the last night before the downtime sequence.
EARLY_OPS_WINDOW = {"Early Ops, Oct–Nov 2025": (ref.DAY_OBS_EARLY_OPS_START, 20251130)}
# LSST on sky: season start to the last night with survey visits (storm from 20260714).
LSST_WINDOW = {"LSST": (ref.DAY_OBS_LSST_START, ref.DAY_OBS_LAST_BEFORE_STORM - 1)}
# Colours per period for the two-point version: Early Ops keeps the teal, LSST is purple.
COLORS = None
LABEL_OFFSETS = None
LABELS_IN_LEGEND = True  # two-point version: describe each window in the legend, no labels next to the points

tag = f"{DAY_OBS_MIN}_{DAY_OBS_MAX}"
nightly = pd.read_parquet(data.DATA_DIR / f"nightly_{tag}.parquet")
if PERIOD == "early_ops":
    (lo, hi), = EARLY_OPS_WINDOW.values()
    visits = pd.read_parquet(data.DATA_DIR / f"visits_{tag}.parquet")
    periods = binning.window_table(visits, nightly, EARLY_OPS_WINDOW)
    stem = f"early_ops_octnov_{lo}-{hi}"
    periods.to_parquet(data.DATA_DIR / f"window_early_ops_octnov_{lo}_{hi}.parquet")
    TITLE = TITLE or "Early Ops, 24 Oct – 30 Nov 2025"
elif PERIOD == "early_ops_vs_lsst":
    (lo, _), = EARLY_OPS_WINDOW.values()
    (_, hi), = LSST_WINDOW.values()
    visits = pd.read_parquet(data.DATA_DIR / f"visits_{tag}.parquet")
    (early_lo_hi,) = EARLY_OPS_WINDOW.values()
    periods = binning.window_table(visits, nightly, {"Early Ops": early_lo_hi, **LSST_WINDOW})
    stem = f"early_ops_vs_lsst_{lo}-{hi}"
    periods.to_parquet(data.DATA_DIR / f"windows_early_ops_vs_lsst_{lo}_{hi}.parquet")
    periods.to_csv(data.DATA_DIR / f"windows_early_ops_vs_lsst_{lo}_{hi}.csv", index=False)
    TITLE = TITLE or "Early Ops (Oct–Nov 2025) vs. LSST (Jun–Jul 2026)"
    COLORS = [rp.TEAL, rp.CATEGORICAL[2]]  # teal, purple
    LABEL_OFFSETS = {0: (9, 9), 1: (-9, -14)}  # Early Ops above-right, LSST below-left
else:
    TABLE_NAME = {"window": "windows", "quarter": "quarterly", "month": "monthly"}[PERIOD]
    periods = pd.read_parquet(data.DATA_DIR / f"{TABLE_NAME}_{tag}.parquet")
    stem = f"{TABLE_NAME}_{DAY_OBS_MIN}-{DAY_OBS_MAX}"
periods[["label_long", "n_nights_calendar", "n_nights_with_visits", "n_visits", "n_vis_expected", "ratio", f"{IQ_COL}_med"]].round(3)

In [ ]:
fig, ax = static.build_static_figure(
    periods,
    nightly,
    iq_col=IQ_COL,
    figsize=FIGSIZE,
    label_size=LABEL_SIZE,
    transparent=TRANSPARENT,
    title=TITLE,
    colors=COLORS,
    label_offsets=LABEL_OFFSETS,
    labels_in_legend=LABELS_IN_LEGEND and PERIOD == "early_ops_vs_lsst",
)
out = static.save_png(
    fig,
    data.DATA_DIR.parent / "figures" / f"speed_vs_iq_static_{stem}.png",
    transparent=TRANSPARENT,
)
print(out)